# 01 — Exploratory Data Analysis: oHCM Cohort & Camzyos Adoption

**IC question:** Who are the oHCM patients in this dataset, who is initiating Camzyos, and what does their symptom profile look like?

This notebook establishes:
1. oHCM cohort identification under two definitions (loose vs. strict)
2. Mavacamten prescription patterns — frequency, persistence, fill gaps
3. Temporal adoption curve
4. Symptom burden among Camzyos patients (dyspnea, HF, fatigue, cardiology E&M)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 120

DATA_DIR = Path("../synthetic_data")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [ ]:
patients = pd.read_csv(DATA_DIR / "patients.csv")
diagnoses = pd.read_csv(DATA_DIR / "diagnoses.csv", parse_dates=["date"])
procedures = pd.read_csv(DATA_DIR / "procedures.csv", parse_dates=["date"])
prescriptions = pd.read_csv(DATA_DIR / "prescriptions.csv", parse_dates=["date"])
enrollment = pd.read_csv(DATA_DIR / "enrollment.csv")
code_dict = pd.read_csv(DATA_DIR / "code_dictionary.csv")

print(f"Patients:      {len(patients):,}")
print(f"Diagnoses:     {len(diagnoses):,}")
print(f"Procedures:    {len(procedures):,}")
print(f"Prescriptions: {len(prescriptions):,}")
print(f"Enrollment:    {len(enrollment):,}")
print(f"Code dict:     {len(code_dict):,}")

---
## 1. oHCM Cohort Identification

Two definitions, compared side-by-side:

| Definition | Rule |
|---|---|
| **Loose** | ≥1 claim with ICD-10 I42.1 |
| **Strict** | (≥2 claims with I42.1, ≥30 days apart) OR (≥1 I42.2 + ≥1 I42.1) |

The strict definition reduces false positives from rule-out diagnoses or coding errors.

In [ ]:
# --- Loose definition: >= 1 I421 claim ---
ohcm_dx = diagnoses[diagnoses["dx_code"] == "I421"].copy()
cohort_loose = set(ohcm_dx["patient_id"].unique())
print(f"Loose cohort (≥1 I421 claim): {len(cohort_loose):,} patients")

In [ ]:
# --- Strict definition ---
# Path A: ≥2 I421 claims, ≥30 days apart
i421_claims = ohcm_dx.sort_values(["patient_id", "date"])
i421_claims["prev_date"] = i421_claims.groupby("patient_id")["date"].shift(1)
i421_claims["days_gap"] = (i421_claims["date"] - i421_claims["prev_date"]).dt.days

path_a = set(i421_claims.loc[i421_claims["days_gap"] >= 30, "patient_id"].unique())
print(f"Path A (≥2 I421, ≥30d apart): {len(path_a):,} patients")

# Path B: ≥1 I422 AND ≥1 I421
has_i422 = set(diagnoses.loc[diagnoses["dx_code"] == "I422", "patient_id"].unique())
has_i421 = cohort_loose
path_b = has_i422 & has_i421
print(f"Path B (I422 + I421):          {len(path_b):,} patients")

cohort_strict = path_a | path_b
print(f"\nStrict cohort (A ∪ B):         {len(cohort_strict):,} patients")
print(f"Loose − Strict (dropped):      {len(cohort_loose - cohort_strict):,} patients")

In [ ]:
# --- How many Camzyos patients does each definition capture? ---
mav_patients = set(
    prescriptions.loc[prescriptions["rx_code"] == "Mavacamten", "patient_id"].unique()
)
print(f"Total Mavacamten patients: {len(mav_patients)}")
print(f"  In loose cohort:  {len(mav_patients & cohort_loose)} / {len(mav_patients)}")
print(f"  In strict cohort: {len(mav_patients & cohort_strict)} / {len(mav_patients)}")
print(f"  Neither:          {len(mav_patients - cohort_loose)} (may have I422/I429 only)")

# What dx codes do Camzyos patients outside loose cohort have?
mav_no_i421 = mav_patients - cohort_loose
if mav_no_i421:
    outside_dx = (
        diagnoses[diagnoses["patient_id"].isin(mav_no_i421)]
        .merge(code_dict, left_on="dx_code", right_on="code", how="left")
        .query("dx_code.str.startswith('I42')")[["dx_code", "description"]]
        .drop_duplicates()
    )
    print(f"\nCardiomyopathy codes for {len(mav_no_i421)} Camzyos patients without I421:")
    display(outside_dx)

In [ ]:
# --- Demographics comparison: loose vs strict ---
def cohort_summary(patient_ids, label):
    df = patients[patients["patient_id"].isin(patient_ids)].copy()
    df["age_2022"] = 2022 - df["birth_year"]
    return pd.Series(
        {
            "label": label,
            "n": len(df),
            "age_median": df["age_2022"].median(),
            "age_q25": df["age_2022"].quantile(0.25),
            "age_q75": df["age_2022"].quantile(0.75),
            "pct_female": (df["sex"] == "F").mean() * 100,
        }
    )


summary = pd.DataFrame(
    [
        cohort_summary(cohort_loose, "Loose (≥1 I421)"),
        cohort_summary(cohort_strict, "Strict (≥2 I421 30d+ OR I422+I421)"),
        cohort_summary(mav_patients, "Mavacamten patients"),
        cohort_summary(cohort_loose - mav_patients, "Loose, no Mavacamten"),
    ]
).set_index("label")

print("Cohort demographics (age as of 2022):")
display(summary)

---
## 2. Mavacamten Prescription Patterns

For each Camzyos patient: how many fills, what supply (30d vs 90d), and are there gaps suggesting discontinuation?

In [ ]:
mav_rx = prescriptions[prescriptions["rx_code"] == "Mavacamten"].copy()
mav_rx = mav_rx.sort_values(["patient_id", "date"])

print(f"Total Mavacamten fills: {len(mav_rx):,}")
print(f"Unique patients:       {mav_rx['patient_id'].nunique()}")
print("\nDays supply distribution:")
print(mav_rx["days_supply"].value_counts().to_string())

In [ ]:
# --- Per-patient fill summary ---
mav_per_patient = (
    mav_rx.groupby("patient_id")
    .agg(
        n_fills=("date", "count"),
        first_fill=("date", "min"),
        last_fill=("date", "max"),
        pct_90d=("days_supply", lambda x: (x == 90).mean() * 100),
    )
    .reset_index()
)
mav_per_patient["duration_days"] = (
    mav_per_patient["last_fill"] - mav_per_patient["first_fill"]
).dt.days

print("Fills per patient:")
display(mav_per_patient["n_fills"].describe().to_frame().T)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(
    mav_per_patient["n_fills"],
    bins=range(1, mav_per_patient["n_fills"].max() + 2),
    edgecolor="white",
    alpha=0.8,
)
axes[0].set_xlabel("Number of fills")
axes[0].set_ylabel("Patients")
axes[0].set_title("Mavacamten fills per patient")

axes[1].hist(
    mav_per_patient.loc[mav_per_patient["duration_days"] > 0, "duration_days"],
    bins=30,
    edgecolor="white",
    alpha=0.8,
)
axes[1].set_xlabel("Days (first fill → last fill)")
axes[1].set_ylabel("Patients")
axes[1].set_title("Treatment duration (patients with ≥2 fills)")

plt.tight_layout()
plt.show()

In [ ]:
# --- Fill gap analysis ---
# For each consecutive fill pair, compute: gap = next_fill_date - (prev_fill_date + days_supply)
# Positive gap = days without medication coverage
mav_rx_sorted = mav_rx.sort_values(["patient_id", "date"]).copy()
mav_rx_sorted["coverage_end"] = mav_rx_sorted["date"] + pd.to_timedelta(
    mav_rx_sorted["days_supply"], unit="D"
)
mav_rx_sorted["next_fill"] = mav_rx_sorted.groupby("patient_id")["date"].shift(-1)
mav_rx_sorted["gap_days"] = (mav_rx_sorted["next_fill"] - mav_rx_sorted["coverage_end"]).dt.days

gaps = mav_rx_sorted.dropna(subset=["gap_days"]).copy()

print(f"Total fill-to-fill intervals: {len(gaps)}")
print("\nGap days (next_fill - coverage_end):")
display(gaps["gap_days"].describe().to_frame().T)

print(
    f"\nGaps > 30 days (potential discontinuation): {(gaps['gap_days'] > 30).sum()} / {len(gaps)} intervals"
)
print(f"Gaps > 60 days: {(gaps['gap_days'] > 60).sum()}")
print(f"Negative gaps (early refill / overlap): {(gaps['gap_days'] < 0).sum()}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(gaps["gap_days"].clip(-30, 120), bins=60, edgecolor="white", alpha=0.8)
ax.axvline(0, color="red", linestyle="--", alpha=0.6, label="Coverage end = next fill")
ax.axvline(30, color="orange", linestyle="--", alpha=0.6, label="30-day gap")
ax.set_xlabel("Gap days (next fill − coverage end)")
ax.set_ylabel("Fill intervals")
ax.set_title("Mavacamten fill gaps: coverage continuity")
ax.legend()
plt.tight_layout()
plt.show()

---
## 3. Mavacamten Prescriptions Over Time

Two views:
- **New initiations per month** (first fill per patient) — the actual adoption signal
- **Total fills per month** — confounds new starts with persistence/refills

In [ ]:
mav_rx["year_month"] = mav_rx["date"].dt.to_period("M")

# Total fills per month
total_fills = mav_rx.groupby("year_month").size().rename("total_fills")

# New initiations per month (first fill per patient)
first_fills = mav_rx.groupby("patient_id")["date"].min().reset_index()
first_fills["year_month"] = first_fills["date"].dt.to_period("M")
new_starts = first_fills.groupby("year_month").size().rename("new_initiations")

# Cumulative initiations
cumulative = new_starts.cumsum().rename("cumulative_patients")

monthly = pd.concat([total_fills, new_starts, cumulative], axis=1).fillna(0).astype(int)
print("Monthly Mavacamten summary:")
display(monthly)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# New initiations
x = range(len(monthly))
labels = [str(p) for p in monthly.index]

axes[0].bar(x, monthly["new_initiations"], alpha=0.7, edgecolor="white")
axes[0].set_xticks(x[::3])
axes[0].set_xticklabels(labels[::3], rotation=45, ha="right")
axes[0].set_ylabel("Patients")
axes[0].set_title("New Camzyos initiations per month")

# Total fills
axes[1].bar(x, monthly["total_fills"], alpha=0.7, color="C1", edgecolor="white")
axes[1].set_xticks(x[::3])
axes[1].set_xticklabels(labels[::3], rotation=45, ha="right")
axes[1].set_ylabel("Fills")
axes[1].set_title("Total Camzyos fills per month")

# Cumulative
axes[2].plot(x, monthly["cumulative_patients"], marker="o", markersize=4)
axes[2].set_xticks(x[::3])
axes[2].set_xticklabels(labels[::3], rotation=45, ha="right")
axes[2].set_ylabel("Cumulative patients")
axes[2].set_title("Cumulative Camzyos patients")

for ax in axes:
    ax.axvline(x=0, color="grey", linestyle=":", alpha=0.5)

fig.suptitle(
    "Camzyos adoption: new starts are flat-to-declining, total fills grow via persistence",
    fontsize=11,
    y=1.02,
)
plt.tight_layout()
plt.show()

---
## 4. Symptom Burden Among Camzyos Patients

We proxy NYHA II/III symptom severity using claims codes:

| Symptom proxy | ICD-10 codes |
|---|---|
| Dyspnea | R0600, R0602, R0609 |
| Heart failure | I50xx (all) |
| Fatigue | R5383, R531, R5381 |
| Cardiology E&M (high complexity) | 99214, 99215, 99204, 99205 |

In [ ]:
DYSPNEA_CODES = ["R0600", "R0602", "R0609"]
HF_CODES = [c for c in diagnoses["dx_code"].unique() if str(c).startswith("I50")]
FATIGUE_CODES = ["R5383", "R531", "R5381"]
CARDIO_EM_CODES = ["99214", "99215", "99204", "99205"]

print(f"HF codes in data: {sorted(HF_CODES)}")
print(f"Dyspnea codes:    {DYSPNEA_CODES}")
print(f"Fatigue codes:    {FATIGUE_CODES}")
print(f"Cardio E&M codes: {CARDIO_EM_CODES}")

In [ ]:
def symptom_prevalence(patient_ids, label):
    """Compute symptom prevalence for a set of patients."""
    dx_sub = diagnoses[diagnoses["patient_id"].isin(patient_ids)]
    px_sub = procedures[procedures["patient_id"].isin(patient_ids)]
    n = len(patient_ids)

    has_dyspnea = dx_sub[dx_sub["dx_code"].isin(DYSPNEA_CODES)]["patient_id"].nunique()
    has_hf = dx_sub[dx_sub["dx_code"].isin(HF_CODES)]["patient_id"].nunique()
    has_fatigue = dx_sub[dx_sub["dx_code"].isin(FATIGUE_CODES)]["patient_id"].nunique()
    has_cardio_em = px_sub[px_sub["px_code"].isin(CARDIO_EM_CODES)]["patient_id"].nunique()

    return pd.Series(
        {
            "group": label,
            "n": n,
            "dyspnea_n": has_dyspnea,
            "dyspnea_pct": has_dyspnea / n * 100,
            "hf_n": has_hf,
            "hf_pct": has_hf / n * 100,
            "fatigue_n": has_fatigue,
            "fatigue_pct": has_fatigue / n * 100,
            "cardio_em_n": has_cardio_em,
            "cardio_em_pct": has_cardio_em / n * 100,
            "any_symptom_n": len(
                set(
                    dx_sub[dx_sub["dx_code"].isin(DYSPNEA_CODES + HF_CODES + FATIGUE_CODES)][
                        "patient_id"
                    ].unique()
                )
                | set(px_sub[px_sub["px_code"].isin(CARDIO_EM_CODES)]["patient_id"].unique())
            ),
            "any_symptom_pct": len(
                set(
                    dx_sub[dx_sub["dx_code"].isin(DYSPNEA_CODES + HF_CODES + FATIGUE_CODES)][
                        "patient_id"
                    ].unique()
                )
                | set(px_sub[px_sub["px_code"].isin(CARDIO_EM_CODES)]["patient_id"].unique())
            )
            / n
            * 100,
        }
    )


symptom_df = pd.DataFrame(
    [
        symptom_prevalence(mav_patients, "Camzyos patients"),
        symptom_prevalence(cohort_loose - mav_patients, "oHCM (loose), no Camzyos"),
        symptom_prevalence(cohort_strict - mav_patients, "oHCM (strict), no Camzyos"),
    ]
).set_index("group")

print("Symptom prevalence (ever, across full observation period):")
display(symptom_df.round(1))

In [ ]:
# --- Side-by-side symptom prevalence ---
pct_cols = ["dyspnea_pct", "hf_pct", "fatigue_pct", "cardio_em_pct"]
labels_nice = ["Dyspnea", "Heart failure", "Fatigue", "High-complexity\ncardiology E&M"]

plot_data = symptom_df[pct_cols].copy()
plot_data.columns = labels_nice

ax = plot_data.T.plot.barh(figsize=(10, 5), edgecolor="white", alpha=0.85)
ax.set_xlabel("% of patients with ≥1 claim")
ax.set_title("Symptom burden: Camzyos patients vs oHCM non-initiators")
ax.legend(title="Group", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# --- Symptom claim frequency (intensity, not just prevalence) ---
# Among patients who HAVE the symptom, how many claims?
def symptom_intensity(patient_ids, codes, source="dx"):
    if source == "dx":
        sub = diagnoses[
            (diagnoses["patient_id"].isin(patient_ids)) & (diagnoses["dx_code"].isin(codes))
        ]
        return sub.groupby("patient_id").size()
    else:
        sub = procedures[
            (procedures["patient_id"].isin(patient_ids)) & (procedures["px_code"].isin(codes))
        ]
        return sub.groupby("patient_id").size()


print("Claim frequency per patient (among those with ≥1 claim):")
print()
for group_ids, group_name in [
    (mav_patients, "Camzyos"),
    (cohort_loose - mav_patients, "oHCM no-Camzyos"),
]:
    print(f"--- {group_name} ---")
    for codes, code_label, src in [
        (DYSPNEA_CODES, "Dyspnea", "dx"),
        (HF_CODES, "Heart failure", "dx"),
        (FATIGUE_CODES, "Fatigue", "dx"),
        (CARDIO_EM_CODES, "Cardio E&M", "px"),
    ]:
        counts = symptom_intensity(group_ids, codes, source=src)
        if len(counts) > 0:
            print(
                f"  {code_label:20s}  n={len(counts):4d}  median={counts.median():.0f}  mean={counts.mean():.1f}  max={counts.max()}"
            )
        else:
            print(f"  {code_label:20s}  n=0")
    print()

---
## Summary

Key findings to carry forward:

---
## 5. Enrollment Windows Around Camzyos Prescriptions

Critical for survival analysis: how much observation time do we have before and after each patient's Camzyos exposure?

- **Pre-first-fill enrollment**: How long was a patient enrolled before their first Camzyos fill? Short windows mean we can't observe prior treatment history reliably.
- **Post-last-fill enrollment**: How long do we observe a patient after their last fill? Short windows = right-censoring that may disguise discontinuation.

In [ ]:
# --- Compute enrollment spans for Camzyos patients ---
# enrollment has (patient_id, month, enrolled) where month is YYYY-MM
enr_mav = enrollment[
    (enrollment["patient_id"].isin(mav_patients)) & (enrollment["enrolled"] == 1)
].copy()

# Parse month to first-of-month date for arithmetic
enr_mav["month_date"] = pd.to_datetime(enr_mav["month"] + "-01")

# Per-patient: first and last enrolled month
enr_spans = (
    enr_mav.groupby("patient_id")["month_date"].agg(enr_first="min", enr_last="max").reset_index()
)

# Merge with first/last Camzyos fill
enr_spans = enr_spans.merge(
    mav_per_patient[["patient_id", "first_fill", "last_fill"]],
    on="patient_id",
    how="inner",
)

# Compute months before first fill and after last fill (using 30.44 days/month)
enr_spans["months_enrolled_before_first_fill"] = (
    (enr_spans["first_fill"] - enr_spans["enr_first"]).dt.days / 30.44
).round(1)

enr_spans["months_enrolled_after_last_fill"] = (
    (enr_spans["enr_last"] - enr_spans["last_fill"]).dt.days / 30.44
).round(1)

print(f"Camzyos patients with enrollment data: {len(enr_spans)}")
print("\nMonths enrolled BEFORE first Camzyos fill:")
display(enr_spans["months_enrolled_before_first_fill"].describe().to_frame().T.round(1))
print("\nMonths enrolled AFTER last Camzyos fill:")
display(enr_spans["months_enrolled_after_last_fill"].describe().to_frame().T.round(1))

In [ ]:
# --- Histograms ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pre-first-fill
axes[0].hist(
    enr_spans["months_enrolled_before_first_fill"],
    bins=30,
    edgecolor="white",
    alpha=0.8,
    color="C0",
)
axes[0].axvline(
    enr_spans["months_enrolled_before_first_fill"].median(),
    color="red",
    linestyle="--",
    alpha=0.7,
    label=f"Median: {enr_spans['months_enrolled_before_first_fill'].median():.1f} mo",
)
axes[0].axvline(6, color="orange", linestyle=":", alpha=0.7, label="6-month minimum (plan)")
axes[0].set_xlabel("Months enrolled before first Camzyos fill")
axes[0].set_ylabel("Patients")
axes[0].set_title("Pre-initiation enrollment window")
axes[0].legend(fontsize=9)

# Post-last-fill
axes[1].hist(
    enr_spans["months_enrolled_after_last_fill"], bins=30, edgecolor="white", alpha=0.8, color="C1"
)
axes[1].axvline(
    enr_spans["months_enrolled_after_last_fill"].median(),
    color="red",
    linestyle="--",
    alpha=0.7,
    label=f"Median: {enr_spans['months_enrolled_after_last_fill'].median():.1f} mo",
)
axes[1].set_xlabel("Months enrolled after last Camzyos fill")
axes[1].set_ylabel("Patients")
axes[1].set_title("Post-last-fill enrollment window")
axes[1].legend(fontsize=9)

fig.suptitle("Enrollment windows around Camzyos prescriptions (n=166)", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

# Flag patients with short windows
short_pre = (enr_spans["months_enrolled_before_first_fill"] < 6).sum()
short_post = (enr_spans["months_enrolled_after_last_fill"] < 1).sum()
print(
    f"\nPatients with <6 months enrollment before first fill: {short_pre} / {len(enr_spans)} ({short_pre / len(enr_spans) * 100:.1f}%)"
)
print(
    f"Patients with <1 month enrollment after last fill:    {short_post} / {len(enr_spans)} ({short_post / len(enr_spans) * 100:.1f}%)"
)
print(
    "  → These patients may have disenrolled shortly after their last fill — censored, not discontinued"
)

In [ ]:
# --- Check for enrollment gaps (intermittent enrollment) among Camzyos patients ---
# Are there months between enr_first and enr_last where enrolled=0?
enr_mav_all = enrollment[enrollment["patient_id"].isin(mav_patients)].copy()
enr_mav_all["month_date"] = pd.to_datetime(enr_mav_all["month"] + "-01")

gap_patients = []
for pid in mav_patients:
    pat = enr_mav_all[enr_mav_all["patient_id"] == pid].sort_values("month_date")
    enrolled_months = pat[pat["enrolled"] == 1]["month_date"]
    if len(enrolled_months) < 2:
        continue
    first, last = enrolled_months.min(), enrolled_months.max()
    in_window = pat[(pat["month_date"] >= first) & (pat["month_date"] <= last)]
    n_gaps = (in_window["enrolled"] == 0).sum()
    if n_gaps > 0:
        gap_patients.append(
            {"patient_id": pid, "gap_months": n_gaps, "enr_first": first, "enr_last": last}
        )

gap_df = pd.DataFrame(gap_patients)
print(
    f"Camzyos patients with enrollment gaps between first and last enrolled month: {len(gap_df)} / {len(mav_patients)}"
)
if len(gap_df) > 0:
    print("Gap months distribution:")
    display(gap_df["gap_months"].describe().to_frame().T)
else:
    print("→ No intermittent enrollment among Camzyos patients — clean continuous enrollment")

In [ ]:
print("=" * 60)
print("EDA SUMMARY")
print("=" * 60)
print("")
print("COHORT SIZES")
print(f"  Loose oHCM (≥1 I421):    {len(cohort_loose):,}")
print(f"  Strict oHCM:             {len(cohort_strict):,}")
print(f"  Camzyos patients:        {len(mav_patients)}")
print(f"  Camzyos in loose cohort: {len(mav_patients & cohort_loose)}")
print(f"  Camzyos in strict cohort:{len(mav_patients & cohort_strict)}")
print("")
print("PRESCRIPTION PATTERNS")
print(f"  Median fills/patient:    {mav_per_patient['n_fills'].median():.0f}")
print(f"  First fill date:         {mav_per_patient['first_fill'].min().date()}")
print(
    f"  Monthly new starts (2023 H2): ~{monthly.loc[monthly.index >= '2023-07', 'new_initiations'].mean():.1f}/month"
)
print("")
print("KEY OBSERVATIONS")
print("  → New initiations flat-to-declining; total fills grow via persistence")
print("  → Strict cohort drops some patients — check if it drops Camzyos patients")
print("  → Symptom burden comparison above shows whether Camzyos patients")
print("    are systematically more symptomatic")